# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path.cwd()
if (ROOT / 'src').exists(): sys.path.insert(0, str(ROOT / 'src'))
print('Root:', ROOT)

In [ ]:
RUN_PROFILE = os.environ.get('CAMERA_DISCOVERY_PROFILE', 'fast')
USER_QUERY = os.environ.get('CAMERA_DISCOVERY_QUERY', 'Get me all cameras from Greenville, Texas')
# Multi-location example:
# USER_QUERY = 'Get me all cameras from London, England and New York, New York'
OUTPUT_DIR = Path(os.environ.get('CAMERA_DISCOVERY_OUTPUT_DIR', 'runs/notebook-live-test'))

os.environ.setdefault('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama')
os.environ.setdefault('CAMERA_DISCOVERY_TARGET_INTENT_MODEL', 'gemma3:4b-cloud')
os.environ.setdefault('CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL', 'gemma3:4b-cloud')
os.environ.setdefault('CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL', 'gemma3:4b-cloud')

DISCOVERY_MODE = os.environ.get('CAMERA_DISCOVERY_DISCOVERY_MODE', 'both')
SOURCES_FILE = Path(os.environ.get('CAMERA_DISCOVERY_SOURCES_FILE', 'SOURCES.md'))
SEED_URLS = [url.strip() for url in os.environ.get('CAMERA_DISCOVERY_SEED_URLS', '').split(',') if url.strip()]

print('profile:', RUN_PROFILE)
print('query:', USER_QUERY)
print('output:', OUTPUT_DIR)
print('provider:', os.environ.get('CAMERA_DISCOVERY_LLM_PROVIDER'))
print('target intent model:', os.environ.get('CAMERA_DISCOVERY_TARGET_INTENT_MODEL'))
print('discovery mode:', DISCOVERY_MODE)
print('sources file:', SOURCES_FILE)
print('seed urls:', len(SEED_URLS))


In [ ]:
cmd = [
    sys.executable, '-m', 'camera_discovery.cli', 'run', USER_QUERY,
    '--profile', RUN_PROFILE,
    '--output-dir', str(OUTPUT_DIR),
    '--discovery-mode', DISCOVERY_MODE,
    '--sources-file', str(SOURCES_FILE),
]
for url in SEED_URLS:
    cmd.extend(['--seed-url', url])
print('$', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
print('exit:', result.returncode)
if result.returncode != 0: raise RuntimeError('camera-discovery run failed')


In [ ]:
from pathlib import Path
import json

for rel in ['logs/source_policy_summary.json', 'logs/candidate_discovery_summary.json', 'logs/run_summary.json']:
    path = OUTPUT_DIR / rel
    print('---', rel, 'exists=', path.exists())
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2)[:4000])


In [ ]:
summary_path = OUTPUT_DIR / 'logs' / 'run_summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
targets = summary.get('targets', [])
print('targets:', len(targets))
for t in targets:
    print(json.dumps({
        'target_id': t.get('target_id'),
        'target_label': t.get('target_label'),
        'canonical_target': t.get('canonical_target'),
        'geometry_status': t.get('geometry_status'),
        'bbox_verified': t.get('bbox_verified'),
        'trust_policy': t.get('trust_policy'),
    }, indent=2))
print(json.dumps({
    'unique_candidates': len(summary.get('candidates',{}).get('unique', [])),
    'coordinate_bearing': len(summary.get('candidates',{}).get('coordinate_bearing', [])),
    'trusted_geojson_features': summary.get('outputs',{}).get('trusted_geojson_features_written'),
    'untrusted_geojson_features': summary.get('outputs',{}).get('untrusted_geojson_features_written'),
}, indent=2))


In [ ]:
for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'review_artifacts.zip',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/geocoder_candidate_scores.json',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


## Camera URL table

This cell loads `camera.geojson` first, then falls back to `untrusted_camera_candidates.geojson` or `untrusted_camera.geojson`. It displays URL, location, coordinate, trust, validation, and source metadata in a table and writes a CSV copy into the run directory.

In [ ]:
from pathlib import Path
from IPython.display import display
from camera_discovery.utils.geojson_viewer import (
    load_camera_rows,
    select_camera_geojson,
    write_camera_table_csv,
)

GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
print('Selected GeoJSON:', GEOJSON_PATH)

if GEOJSON_PATH is None:
    CAMERA_ROWS = []
    print('No trusted or untrusted camera GeoJSON found yet.')
else:
    CAMERA_ROWS = load_camera_rows(GEOJSON_PATH)
    table_csv = write_camera_table_csv(OUTPUT_DIR, CAMERA_ROWS)
    print('Rows:', len(CAMERA_ROWS))
    print('CSV table:', table_csv)
    try:
        import pandas as pd
        columns = [
            'name', 'target_label', 'location_text', 'latitude', 'longitude',
            'stream_url', 'source_url', 'thumbnail_url', 'trust_level',
            'validation_status', 'scope_status', 'discovery_method', 'review_required'
        ]
        df = pd.DataFrame(CAMERA_ROWS)
        display(df[[c for c in columns if c in df.columns]])
    except Exception as exc:
        print('Pandas display unavailable; showing first rows as dictionaries:', repr(exc))
        for row in CAMERA_ROWS[:10]:
            print(row)


## Interactive camera map

The map below embeds the selected GeoJSON directly into the HTML so it works inside Colab. Click a marker to see camera metadata. If a thumbnail/snapshot URL is present in the GeoJSON properties, the popup shows it. The **Play video** button attempts to play the stream URL with hls.js or native browser video support.

In [ ]:
from IPython.display import HTML, display
from camera_discovery.utils.geojson_viewer import write_embedded_camera_map

MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, GEOJSON_PATH, output_name='notebook_camera_map.html')
print('Notebook map:', MAP_PATH)
display(HTML(MAP_PATH.read_text(encoding='utf-8')))
